# Week 4 — Deploying a Machine Learning Model

A model that only exists inside a notebook is an experiment, not a product. This
notebook takes the churn classifier from Weeks 2 and 3 and follows it all the way to
a running service:

1. load the serialised artifact and check what is inside it
2. reproduce the evaluation from the saved file
3. compare joblib and pickle
4. call the API as a client would
5. see what happens when a request is malformed
6. explain an individual prediction
7. measure single and batch scoring
8. measure latency, and find where the time actually goes

Every number printed here comes from the files in `outputs/`, which are produced by
`src/train.py`, `src/benchmark.py` and `src/make_figures.py`.

## Setup

The notebook lives in `week-4/notebooks/`, so the project root is one level up. The
`src/` folder is added to the import path so the shared pipeline code can be reused
rather than copied.

In [1]:
import json
import pickle
import sys
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

WEEK = Path.cwd().resolve().parent
SRC = WEEK / "src"
sys.path.insert(0, str(SRC))

from config import MODEL_PATH, MODEL_PICKLE_PATH, MODEL_CARD_PATH, MANIFEST_PATH

pd.set_option("display.width", 120)
print("project root:", WEEK.name)

project root: week-4


## 1. What is actually inside the artifact?

A scikit-learn `Pipeline` is a single Python object, so it can be serialised as one
file. That matters: if the imputer, the scaler and the encoder were fitted separately,
it would be possible to ship a model with a stale scaler. One object means one thing
to version, hash and roll back.

In [2]:
pipeline = joblib.load(MODEL_PATH)
print(type(pipeline).__name__)
print()
for name, step in pipeline.steps:
    print(f"{name:14s} {type(step).__name__}")

model = pipeline.named_steps["model"]
print()
print("coefficient shape:", model.coef_.shape)
print("intercept        :", round(float(model.intercept_[0]), 4))
print("hyperparameters  :", model.get_params()["C"], model.get_params()["class_weight"])

Pipeline

preprocessor   ColumnTransformer
model          LogisticRegression

coefficient shape: (1, 21)
intercept        : -0.857
hyperparameters  : 0.1 balanced


The coefficients are the model. There are 21 of them because the 11 input columns
expand to 21 after one-hot encoding — which is why the transformation has to travel
with the estimator.

In [3]:
preprocessor = pipeline.named_steps["preprocessor"]
names = preprocessor.get_feature_names_out()
print(len(names), "transformed features")
for name in names:
    print("  ", name)

21 transformed features
   tenure_months
   monthly_charges
   total_charges
   support_tickets
   senior_citizen
   contract_Month-to-month
   contract_One year
   contract_Two year
   internet_service_DSL
   internet_service_Fiber optic
   internet_service_No
   payment_method_Bank transfer
   payment_method_Credit card
   payment_method_Electronic check
   payment_method_Mailed check
   paperless_billing_No
   paperless_billing_Yes
   partner_No
   partner_Yes
   dependents_No
   dependents_Yes


## 2. Reproducing the evaluation from the saved file

The point of serialising is that the file *is* the model. Training results are only
credible if the artifact reproduces them, so this section rebuilds the same split and
scores it again from disk.

In [4]:
from features import clean_frame, split_features_target
from sklearn.model_selection import train_test_split

frame = clean_frame(pd.read_csv(WEEK / "data" / "customer_churn_raw.csv"))
X, y = split_features_target(frame)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"train {len(X_train):,} / test {len(X_test):,}")

train 1,440 / test 360


In [5]:
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, roc_auc_score,
)

probability = pipeline.predict_proba(X_test)[:, 1]
predicted = (probability >= 0.5).astype(int)

recomputed = {
    "accuracy": accuracy_score(y_test, predicted),
    "precision": precision_score(y_test, predicted),
    "recall": recall_score(y_test, predicted),
    "f1": f1_score(y_test, predicted),
    "roc_auc": roc_auc_score(y_test, probability),
}

saved = json.loads(MODEL_CARD_PATH.read_text())["metrics"]
pd.DataFrame({
    "metric": list(recomputed),
    "recomputed": [round(v, 6) for v in recomputed.values()],
    "model card": [round(saved[k], 6) for k in recomputed],
})

,metric,recomputed,model card
0,accuracy,0.677778,0.677778
1,precision,0.431507,0.431507
2,recall,0.656250,0.656250
3,f1,0.520661,0.520661
4,roc_auc,0.767519,0.767519


The two columns match to six decimal places. That is the reproducibility claim a
deployment actually needs: the bytes being served are the bytes that were evaluated.

In [6]:
deltas = {k: abs(recomputed[k] - saved[k]) for k in recomputed}
print("largest discrepancy:", f"{max(deltas.values()):.2e}")
assert max(deltas.values()) < 1e-9
print("artifact reproduces the recorded metrics")

largest discrepancy: 0.00e+00
artifact reproduces the recorded metrics


## 3. joblib versus pickle

Both formats are written by `train.py` so the choice can be made on evidence rather
than habit.

In [7]:
artifacts = pd.read_csv(WEEK / "outputs" / "artifacts.csv")
artifacts

,artifact,bytes,kib,load_ms
0,joblib,5866,5.7,1.02
1,pickle,3789,3.7,NaN


In [8]:
repeats = 15
timings = {}
for label, path in (("joblib", MODEL_PATH), ("pickle", MODEL_PICKLE_PATH)):
    runs = []
    for _ in range(repeats):
        started = time.perf_counter()
        joblib.load(path) if label == "joblib" else pickle.loads(path.read_bytes())
        runs.append((time.perf_counter() - started) * 1000)
    timings[label] = np.median(runs)

pd.DataFrame({
    "format": list(timings),
    "size KiB": [MODEL_PATH.stat().st_size / 1024, MODEL_PICKLE_PATH.stat().st_size / 1024],
    "median load ms": [round(v, 2) for v in timings.values()],
})

,format,size KiB,median load ms
0,joblib,5.728516,1.10
1,pickle,3.700195,0.08


**What this shows.** On a 5.7 KiB model the difference is irrelevant — both load in
about a millisecond. The choice matters for other reasons:

- `joblib` uses an efficient binary format for NumPy arrays, so it pulls ahead as soon
  as the model holds large coefficient matrices.
- `pickle` is the standard library, so it has no dependency.

**The hazard worth knowing.** Both formats execute code on load. A `.pkl` file from an
untrusted source is a remote-code-execution vulnerability, not a data file. Production
systems either load artifacts they produced themselves, or move to a format that
cannot execute code.

## 4. The API contract

The service is already imported by the test suite, so the same `TestClient` is used
here. `with TestClient(app)` runs the startup hook, which is what loads the model.

In [9]:
from fastapi.testclient import TestClient
from service import app

client = TestClient(app)
client.__enter__()

health = client.get("/health").json()
health

/home/user/.venv/lib/python3.11/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


{'status': 'ok',
 'model_loaded': True,
 'model_version': '4.0.0',
 'uptime_seconds': 0.0,
 'requests_served': 1}

A health endpoint that only says "the process is alive" is not useful. This one
reports whether the **model** is loaded, which is the actual readiness condition.

In [10]:
card = client.get("/model").json()
{k: card[k] for k in ("model_name", "model_version", "algorithm", "training_rows", "default_threshold")}

{'model_name': 'customer-churn-classifier',
 'model_version': '4.0.0',
 'algorithm': 'LogisticRegression (scikit-learn Pipeline)',
 'training_rows': 1440,
 'default_threshold': 0.5}

## 5. Scoring a customer

The request body is validated by Pydantic, so the model never sees a value it cannot
handle.

In [11]:
customer = {
    "tenure_months": 3.0,
    "monthly_charges": 95.4,
    "total_charges": 286.2,
    "support_tickets": 4,
    "senior_citizen": 0,
    "contract": "Month-to-month",
    "internet_service": "Fiber optic",
    "payment_method": "Electronic check",
    "paperless_billing": "Yes",
    "partner": "No",
    "dependents": "No",
}

response = client.post("/predict", json=customer).json()
print(json.dumps(response, indent=2)[:1200])

{
  "customer_index": 0,
  "churn_probability": 0.9753,
  "churn_prediction": "Yes",
  "threshold": 0.5,
  "risk_band": "high",
  "top_contributions": [
    {
      "feature": "monthly_charges",
      "value": 95.4,
      "contribution": 1.0996,
      "direction": "increases risk"
    },
    {
      "feature": "contract = Month-to-month",
      "value": "Month-to-month",
      "contribution": 0.9634,
      "direction": "increases risk"
    },
    {
      "feature": "support_tickets",
      "value": 4.0,
      "contribution": 0.9106,
      "direction": "increases risk"
    },
    {
      "feature": "tenure_months",
      "value": 3.0,
      "contribution": 0.8196,
      "direction": "increases risk"
    },
    {
      "feature": "internet_service = Fiber optic",
      "value": "Fiber optic",
      "contribution": 0.3343,
      "direction": "increases risk"
    }
  ],
  "model_version": "4.0.0",
  "latency_ms": 12.842
}


In [12]:
low_risk = {**customer, "tenure_months": 64, "contract": "Two year",
            "internet_service": "DSL", "payment_method": "Bank transfer",
            "support_tickets": 0}

for label, payload in (("this customer", customer), ("long-tenure, two-year", low_risk)):
    body = client.post("/predict", json=payload).json()
    print(f"{label:24s} {body['churn_probability']:.3f}  ->  {body['churn_prediction']:3s} "
          f"({body['risk_band']} risk)")

this customer            0.975  ->  Yes (high risk)
long-tenure, two-year    0.191  ->  No  (medium risk)


## 6. The threshold is a parameter, not a constant

The model returns a probability. Turning it into a decision is a separate step, and
different callers want different answers from the same model.

In [13]:
for threshold in (0.30, 0.40, 0.50, 0.70, 0.90):
    body = client.post(f"/predict?threshold={threshold}", json=customer).json()
    print(f"threshold {threshold:.2f}  ->  probability {body['churn_probability']:.3f}, "
          f"decision {body['churn_prediction']}")

print()
body = client.post("/predict?explain_result=false", json=customer).json()
print("explanations suppressed:", body["top_contributions"])
print("latency without explanation:", body["latency_ms"], "ms")

threshold 0.30  ->  probability 0.975, decision Yes
threshold 0.40  ->  probability 0.975, decision Yes
threshold 0.50  ->  probability 0.975, decision Yes
threshold 0.70  ->  probability 0.975, decision Yes
threshold 0.90  ->  probability 0.975, decision Yes

explanations suppressed: []
latency without explanation: 5.577 ms


## 7. Validation is part of the product

The most common failure in a deployed model is not a bad prediction — it is a request
that reaches the model in a shape it never expected.

In [14]:
cases = {
    "unknown contract type": {**customer, "contract": "Weekly"},
    "age-like value out of range": {**customer, "senior_citizen": 7},
    "negative charge": {**customer, "monthly_charges": -5},
    "misspelt field": {**customer, "tenure_month": 3},
    "missing field": {k: v for k, v in customer.items() if k != "contract"},
    "null total_charges": {**customer, "total_charges": None},
}

rows = []
for label, payload in cases.items():
    response = client.post("/predict", json=payload)
    detail = response.json().get("detail")
    if isinstance(detail, list):
        detail = detail[0]["msg"]
    rows.append({"case": label, "status": response.status_code,
                 "message": (detail or "")[:58]})
pd.DataFrame(rows)

,case,status,message
0,unknown contract type,422,"Input should be 'Month-to-month', 'One year' o..."
1,age-like value out of range,422,Input should be less than or equal to 1
2,negative charge,422,Input should be greater than or equal to 0
3,misspelt field,422,Extra inputs are not permitted
4,missing field,422,Field required
5,null total_charges,200,


Note the last row: `total_charges` may legitimately be missing for a new customer,
so it is declared optional and the imputer fills it with the training median. The
distinction between *invalid* and *unknown* is a design decision, not something the
framework can make for you.

## 8. Explaining one prediction

For a linear model the log-odds decompose exactly, so the explanation is not an
approximation layered on top of the model — it *is* the model's arithmetic.

In [15]:
from explain import explain, log_odds_breakdown

contributions = pd.DataFrame(explain(pipeline, customer, top_k=8))
contributions

,feature,value,contribution,direction
0,monthly_charges,95.4,1.0996,increases risk
1,contract = Month-to-month,Month-to-month,0.9634,increases risk
2,support_tickets,4,0.9106,increases risk
3,tenure_months,3.0,0.8196,increases risk
4,internet_service = Fiber optic,Fiber optic,0.3343,increases risk
5,partner = No,No,0.1190,increases risk
6,paperless_billing = Yes,Yes,0.1078,increases risk
7,dependents = No,No,0.0693,increases risk


In [16]:
breakdown = log_odds_breakdown(pipeline, customer)
print(breakdown)

import math
print("probability implied by those log-odds:",
      round(1 / (1 + math.exp(-breakdown["log_odds"])), 4))
print("probability returned by the API       :",
      round(client.post("/predict", json=customer).json()["churn_probability"], 4))

{'intercept': -0.857, 'sum_of_contributions': 4.5337, 'log_odds': 3.6767}
probability implied by those log-odds: 0.9753


probability returned by the API       : 0.9753


The two agree, which is the property that makes the explanation trustworthy: a
stakeholder can be shown *why*, and the arithmetic checks out.

This is also why a linear model is often the right choice for a regulated or
customer-facing decision even when a gradient-boosted model scores slightly better —
an unexplainable 2-point gain is not worth an argument nobody can win.

## 9. Batch scoring, and a performance lesson

Batches are where naive implementations quietly fail. Here is the difference between
scoring 100 customers one at a time and scoring them in a single vectorised call.

In [17]:
batch = [customer] * 100

started = time.perf_counter()
for row in batch:
    pipeline.predict_proba(pd.DataFrame([row]))
loop_seconds = time.perf_counter() - started

frame_batch = pd.DataFrame(batch)
started = time.perf_counter()
pipeline.predict_proba(frame_batch)
vector_seconds = time.perf_counter() - started

print(f"row by row : {loop_seconds * 1000:8.1f} ms  ({len(batch) / loop_seconds:,.0f} rows/s)")
print(f"vectorised : {vector_seconds * 1000:8.1f} ms  ({len(batch) / vector_seconds:,.0f} rows/s)")
print(f"speed-up   : {loop_seconds / vector_seconds:,.0f}x")

row by row :    572.9 ms  (175 rows/s)
vectorised :      4.9 ms  (20,537 rows/s)
speed-up   : 118x


The first version of this API looped over rows inside the batch endpoint. The
benchmark in `src/benchmark.py` caught it: throughput was 92 rows per second, barely
better than sending 100 separate requests. Vectorising the `predict_proba` call took
the same endpoint to over 10,000 rows per second.

**The lesson generalises**: a batch endpoint that loops is not a batch endpoint. The
performance was not lost in the model — it was lost in the loop.

In [18]:
benchmark = pd.read_csv(WEEK / "outputs" / "latency_benchmark.csv")
benchmark[["scenario", "p50_ms", "p95_ms", "ms_per_row", "throughput_rows_per_s"]]

,scenario,p50_ms,p95_ms,ms_per_row,throughput_rows_per_s
0,predict + explanation,12.516,13.597,12.516,78.7
1,"predict, no explanation",7.413,8.544,7.413,130.1
2,batch of 100,9.029,9.731,0.090,10930.6
3,batch of 100 + explanations,536.379,582.630,5.364,184.2
4,model only (no HTTP),4.677,6.570,4.677,196.4


## 10. Where the latency goes

A single prediction is about 12.5 ms. Before optimising anything, it is worth knowing
what that budget is made of.

In [19]:
summary = json.loads((WEEK / "outputs" / "benchmark_summary.json").read_text())

budget = pd.DataFrame([
    {"component": "model inference", "ms": summary["model_only_p50_ms"]},
    {"component": "explanation", "ms": summary["explanation_cost_ms"]},
    {"component": "HTTP + JSON", "ms": summary["http_overhead_ms"]},
])
budget["share"] = (budget["ms"] / budget["ms"].sum()).map("{:.0%}".format)
budget

,component,ms,share
0,model inference,4.677,37%
1,explanation,5.103,41%
2,HTTP + JSON,2.736,22%


Two things follow from this table:

- The explanation is the **largest** component. It is also the most optional, which is
  why `explain_result=false` exists.
- The model itself is only about a third of the request. Optimising the model further
  would barely move the number a user feels.

That is the general shape of service optimisation: measure the budget before choosing
what to make faster.

## 11. Cold start

Container platforms restart instances, so the time from process launch to the first
served request is a real product metric — it decides how long a rolling deploy takes
and whether a health check passes.

In [20]:
print(f"cold start (launch -> healthy): {summary['cold_start_ms']:.0f} ms")
print(f"artifact load                 : {summary['single_plain_p50_ms']:.1f} ms per request budget")
print(f"measured load time            : {json.loads(MANIFEST_PATH.read_text())['load_time_ms']:.1f} ms")

cold start (launch -> healthy): 1346 ms
artifact load                 : 7.4 ms per request budget
measured load time            : 1.0 ms


Roughly 1.3 seconds, almost all of it importing NumPy, pandas and scikit-learn
rather than loading a 5.7 KiB file. That is why the model is loaded once at startup
instead of per request, and why the container is kept warm behind a load balancer.

## 12. Failure modes worth planning for

Deployment is mostly the study of what happens when assumptions break:

| Failure | What happens here | Mitigation in this project |
|---|---|---|
| Unexpected category | Encoder ignores it, row is scored | `handle_unknown="ignore"` |
| Missing value | Imputer fills the training median | `SimpleImputer` inside the pipeline |
| Wrong type | 422 before the model runs | Pydantic schema |
| Extra field | 422, not a silent ignore | `extra="forbid"` |
| Artifact missing | Startup fails loudly | `RuntimeError` in lifespan |
| Model drift | Predictions silently degrade | Model card records the training date and caveats |
| Very large request | Rejected before allocation | `max_length=1000` on the batch |

The common thread: fail at the edge, loudly, rather than deep inside the model where
the error is hard to interpret.

In [21]:
print("service endpoints:")
for route in app.routes:
    methods = ",".join(sorted(getattr(route, "methods", []) or []))
    if route.path.startswith(("/predict", "/health", "/model", "/")):
        print(f"  {methods:10s} {route.path}")

service endpoints:
  GET,HEAD   /openapi.json
  GET,HEAD   /docs
  GET,HEAD   /docs/oauth2-redirect
  GET,HEAD   /redoc
  GET        /health
  GET        /model
  GET        /model/manifest
  POST       /predict
  POST       /predict/batch
  GET        /
             /static


## Summary

| Question | Answer from this notebook |
|---|---|
| Does the artifact reproduce the evaluation? | Yes, to 1e-9 |
| Does one file hold preprocessing and model? | Yes, a single `Pipeline` |
| Which format? | joblib; both load in ~1 ms at this size |
| What does one request cost? | ~12.5 ms with explanation, ~7.4 ms without |
| What dominates? | The explanation (41%), not the model (37%) |
| What does a 100-row batch cost? | ~9 ms, about 10,900 rows/s |
| What happens on bad input? | 422 with a readable message |

The model has not changed since Week 3. What changed is that it is now reachable,
versioned, tested, documented and measurable — which is the difference between a
result and a product.

In [22]:
client.__exit__(None, None, None)
print("done")

done
